In [2]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# CARGA DE DATOS 
train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# LLenar los registros vacíos del train
train_data['Text'] = train_data['Text'].fillna('')

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()

In [3]:
# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", strip_accents="unicode", max_features=5000)
X_train = vectorizer.fit_transform(train_data['Text'].values)
X_test = vectorizer.transform(test_data['Text'].values)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

In [4]:
#Spliteo del conjunto de entrenamiento para validación
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


# Multilayer Neural Network Classifier para clasificación multi-etiqueta
base_model = MLPClassifier(max_iter=1000)
mlp_model = MultiOutputClassifier(base_model)
mlp_model.fit(X_train, y_train)

#Predicción
y_pred = mlp_model.predict(X_val)

In [5]:
# EVALUACIÓN

print("\nResultados Neural Network Classifier:\n")
print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_val, y_pred, target_names=emotion_classes, zero_division=0))



Resultados Neural Network Classifier:

Accuracy: 0.28392075558627045

Reporte de clasificación:
                 precision    recall  f1-score   support

    admiration       0.59      0.50      0.54       863
     amusement       0.68      0.54      0.60       453
         anger       0.43      0.29      0.35       323
     annoyance       0.28      0.13      0.18       483
      approval       0.22      0.13      0.17       577
        caring       0.29      0.16      0.21       212
     confusion       0.22      0.12      0.16       258
     curiosity       0.21      0.11      0.15       460
        desire       0.37      0.23      0.28       128
disappointment       0.19      0.06      0.09       244
   disapproval       0.16      0.09      0.12       383
       disgust       0.42      0.27      0.33       156
 embarrassment       0.51      0.31      0.39        58
    excitement       0.33      0.17      0.23       175
          fear       0.48      0.26      0.34       116
     

In [6]:
# MATRICES DE CONFUSIÓN MULTIETIQUETA

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Directorio para los plots
output_dir = "../../Plots/Experiment_Multilabel/Neural_Network/" 

# FUNCIÓN PARA PLOT DE MATRIZ BINARIA POR EMOCIÓN
def plot_confusion_matrix_binary(y_true_col, y_pred_col, label):
    cm = confusion_matrix(y_true_col, y_pred_col, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues", 
        xticklabels=["No", "Sí"],
        yticklabels=["No", "Sí"],
        cbar=True
    )
    plt.title(f"Matriz de Confusión - {label}")
    plt.ylabel("Etiqueta Verdadera")
    plt.xlabel("Etiqueta Predicha")
    plt.tight_layout()
    filename = output_dir + label.lower().replace(" ", "_") + ".png"
    plt.savefig(filename)
    plt.close()

# APLICAR MATRIZ DE CONFUSIÓN PARA CADA EMOCIÓN
for i, emotion in enumerate(emotion_classes):
    y_true_col = y_test[:, i].flatten()
    y_pred_col = y_pred[:, i].flatten()  
    plot_confusion_matrix_binary(y_true_col, y_pred_col, emotion)

ValueError: Found input variables with inconsistent numbers of samples: [5427, 8682]